PART D — Imbalance & Operational Stress
(imbalance = arrivals − departures as defined in PDF)
For Part D, I load the trip data, count how many arrivals and departures each station has, and compute the imbalance using the formula from the PDF:
imbalance = arrivals − departures.
Then I list the stations that keep emptying, the stations that keep overfilling, and the ones that change the most day-to-day.


In [1]:
import pandas as pd

df = pd.read_csv("202501-citibike-tripdata_1.csv")

df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

departures = df.groupby("start_station_name").size().rename("departures")
arrivals = df.groupby("end_station_name").size().rename("arrivals")

imbalance = pd.concat([arrivals, departures], axis=1).fillna(0)
imbalance["imbalance"] = imbalance["arrivals"] - imbalance["departures"]

top_deficits = imbalance.sort_values("imbalance").head(15)
top_surplus = imbalance.sort_values("imbalance", ascending=False).head(15)

df["date"] = df["started_at"].dt.date
daily_counts = df.groupby(["date", "start_station_name"]).size().unstack(fill_value=0)
volatility = daily_counts.std().sort_values(ascending=False)

print("=== Stations That Keep Emptying (Deficits) ===")
print(top_deficits)

print("\n=== Stations That Keep Overfilling (Surplus) ===")
print(top_surplus)

print("\n=== Most Volatile Stations (Day-to-Day Changes) ===")
print(volatility.head(15))


/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_1889/745171076.py:3: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("202501-citibike-tripdata_1.csv")


=== Stations That Keep Emptying (Deficits) ===
                            arrivals  departures  imbalance
1 Ave & E 16 St               1531.0      1834.0     -303.0
Broadway & W 56 St             968.0      1227.0     -259.0
Broadway & E 14 St            2737.0      2994.0     -257.0
W 21 St & 6 Ave               3930.0      4169.0     -239.0
11 Ave & W 41 St              3165.0      3394.0     -229.0
Sterling Pl & New York Ave     318.0       543.0     -225.0
N 7 St & Driggs Ave           2063.0      2283.0     -220.0
Lafayette St & E 8 St         3195.0      3405.0     -210.0
W 44 St & 11 Ave              1725.0      1929.0     -204.0
St Marks Pl & 1 Ave           1393.0      1597.0     -204.0
Ave C & E 16 St                606.0       805.0     -199.0
W 54 St & 9 Ave               1519.0      1703.0     -184.0
6 Ave & W 33 St               2676.0      2856.0     -180.0
W 54 St & 11 Ave              1594.0      1772.0     -178.0
Riverside Dr & W 82 St         627.0       804.0     

PART F — Network Inference & Link Prediction
For Part F, I build a station network from the trip data and then run three link-prediction methods: Adamic–Adar, Preferential Attachment, and Common Neighbors. These scores help me find “emerging” connections between stations, meaning station pairs that might grow into strong corridors in the future even if they don’t currently have many trips.

In [2]:
import pandas as pd
import networkx as nx

df = pd.read_csv("202501-citibike-tripdata_1.csv")

df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

G = nx.DiGraph()

for _, row in df.iterrows():
    u = row["start_station_name"]
    v = row["end_station_name"]
    if pd.isna(u) or pd.isna(v):
        continue
    if G.has_edge(u, v):
        G[u][v]["weight"] += 1
    else:
        G.add_edge(u, v, weight=1)

G_u = nx.Graph()

for u, v, data in G.edges(data=True):
    w = data["weight"]
    if G_u.has_edge(u, v):
        G_u[u][v]["weight"] += w
    else:
        G_u.add_edge(u, v, weight=w)

aa_scores = list(nx.adamic_adar_index(G_u))
aa_sorted = sorted(aa_scores, key=lambda x: x[2], reverse=True)
top_adamic_adar = aa_sorted[:20]

pa_scores = list(nx.preferential_attachment(G_u))
pa_sorted = sorted(pa_scores, key=lambda x: x[2], reverse=True)
top_preferential_attachment = pa_sorted[:20]

cn_scores = list(nx.common_neighbor_centrality(G_u))
cn_sorted = sorted(cn_scores, key=lambda x: x[2], reverse=True)
top_common_neighbors = cn_sorted[:20]

print("=== Top Adamic-Adar Predictions ===")
print(top_adamic_adar)

print("\n=== Top Preferential Attachment Predictions ===")
print(top_preferential_attachment)

print("\n=== Top Common Neighbor Predictions ===")
print(top_common_neighbors)


/var/folders/lt/0y1_b0353bn8xc4j6_sc_chr0000gn/T/ipykernel_1889/3076977882.py:4: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("202501-citibike-tripdata_1.csv")


KeyboardInterrupt: 